# Semiconductor market intelligence: exploratory analysis
This notebook reads the pipeline's real CSV exports. Run `python -m src.pipeline` first. Missing industry or onsemi disclosure data remains missing; charts that need it are skipped. Growth comparisons are descriptive, not causal.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.metrics import company_metrics, market_metrics
exports = ROOT / 'data/exports'
company_path = exports / 'vw_company_financials.csv'
if not company_path.exists():
    raise FileNotFoundError('Run python -m src.pipeline before this notebook')
companies = pd.read_csv(company_path, parse_dates=['period_end'])
markets = pd.read_csv(exports / 'vw_market_sales.csv', parse_dates=['date'])
print(f'{companies.ticker.nunique()} companies, {len(companies)} company-quarter rows, {len(markets)} market rows')

## Revenue trends
The companies have different fiscal calendars. Period-end dates are shown rather than treating fiscal Q1 as the same calendar dates for all firms.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for ticker, group in companies.groupby('ticker'):
    ax.plot(group.period_end, group.revenue / 1e9, label=ticker)
ax.set(title='Quarterly reported revenue', ylabel='USD billions', xlabel='Period end')
ax.legend(ncol=3)
plt.show()

## Margins and R&D
Margins are calculated only where reported revenue is nonzero. Missing gross profit or R&D stays blank.

In [ ]:
financial = company_metrics(companies)
onsemi = financial.loc[financial.ticker.eq('ON')].sort_values('period_end')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(onsemi.period_end, onsemi.gross_margin, label='Gross margin')
axes[0].plot(onsemi.period_end, onsemi.operating_margin, label='Operating margin')
axes[0].legend(); axes[0].set_title('onsemi margins')
axes[1].plot(onsemi.period_end, onsemi.rd_expense / 1e6)
axes[1].set_title('onsemi R&D expense'); axes[1].set_ylabel('USD millions')
plt.tight_layout(); plt.show()

## Industry market and onsemi comparison
The pipeline writes a comparison only when the source has complete matching three-month windows. Growth rates compare different mixes of products and geographies.

In [ ]:
if markets.empty:
    print('No WSTS industry file loaded; market analysis is unavailable.')
else:
    industry = market_metrics(markets)
    world = industry.loc[industry.region.eq('World')]
    if world.empty:
        print('No explicit World series; review four-region coverage before plotting a global total.')
    else:
        ax = world.plot(x='date', y='monthly_sales', figsize=(10, 4), legend=False, title='Global monthly billings')
        ax.set_ylabel('Source file unit'); plt.show()
comparison = pd.read_csv(exports / 'onsemi_industry_comparison.csv')
if comparison.empty:
    print('No complete onsemi / industry comparison windows available.')
else:
    comparison.plot(x='onsemi_period_end', y=['onsemi_yoy_growth', 'industry_yoy_growth'], figsize=(10, 4))
    plt.ylabel('Year-over-year growth, fraction'); plt.show()
    if len(comparison) >= 12:
        print('Descriptive Pearson correlation:', round(comparison[['onsemi_yoy_growth', 'industry_yoy_growth']].corr().iloc[0, 1], 3))
    else:
        print('Correlation omitted: fewer than 12 matched quarters.')